![Gnoffo N2 Enthalpy Fit](gnoffo_n2_enthalpy_fit.png)

In [40]:
A_2 = [
    0.2896319e1,
    0.1515486e-2,
    -0.5723527e-6,
    0.9980739e-10,
    -0.6522355e-14,
    -0.9058620e3,
]
A_3 = [0.3727e1, 0.4684e-3, -0.1140e-6, 0.1154e-10, -0.3293e-15, -0.1043e4]

R_N2 = (
    8.314 / 28 * 1000
)  # universal gas constant / molar mass of N2, J/mol/K / (g/mol) * g/kg = J/kg/K


def h(T, A):
    return R_N2 * (
        (A[0] * T**1) / 1
        + (A[1] * T**2) / 2
        + (A[2] * T**3) / 3
        + (A[3] * T**4) / 4
        + (A[4] * T**5) / 5
        + A[5]
    )

T_ref = 298.16  # K, Gnoffo
print("Vibroelectronic enthalpy from Gnoffo's curve fit")
print(f"T=6000 K in range of 1000 <= T <=  6000: {h(6000, A_2)- 3.5 * R_N2 * (6000-T_ref):.0f} J/kg")
print(f"T=6000 K in range of 6000 <= T <= 15000: {h(6000, A_3)- 3.5 * R_N2 * (6000-T_ref):.0f} J/kg")

Vibroelectronic enthalpy from Gnoffo's curve fit
T=6000 K in range of 1000 <= T <=  6000: 1419020 J/kg
T=6000 K in range of 6000 <= T <= 15000: 1428996 J/kg


In [ ]:
import jax
import jax.numpy as jnp
import plotly.graph_objects as go
from pathlib import Path

from compressible_core import thermodynamic_relations, energy_models
from compressible_core.chemistry_utils import load_species_table
from compressible_core.energy_models_utils import _load_gnoffo_energy_data

jax.config.update("jax_enable_x64", True)

import sys

species_name = "N2"  # enthalpy of this species will be plotted

# Add src directory to path
repo_root = Path.cwd().parent  # Assuming notebook is in experiments/
src_path = repo_root / "src"
sys.path.insert(0, str(src_path))

# Load data
general_species_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/" "air_5_gnoffo.json"
)

gnoffo_equilibrium_enthalpy_data_path = (
    "/home/hhoechter/tum/jaxfluids_internship/data/"
    "air_5_gnoffo_equilibrium_enthalpy.json"
)

energy_model_config_gnoffo = energy_models.EnergyModelConfig(
    model="gnoffo",
    include_electronic=True,
    data_path=gnoffo_equilibrium_enthalpy_data_path,
)

species_table = load_species_table(
    species_names=[species_name],
    general_data_path=general_species_data_path,
    energy_model_config=energy_model_config_gnoffo,
)
# Get species index
species_index = species_table.names.index(species_name)

T_limit_low, T_limit_high, _ = _load_gnoffo_energy_data(
    gnoffo_equilibrium_enthalpy_data_path, [species_name]
)

T = jnp.linspace(
    T_limit_low[0, 0],
    T_limit_high[0, -1],
    35000,
    endpoint=False,
)

# Compute enthalpy
h = thermodynamic_relations.compute_e_ve(
    T_V=T,
    species_table=species_table,
)

h_species = h[species_index, :]

print("Vibroelectronic enthalpy from Gnoffo's curve fit evaluated with repo code")
print(
    f"T=5999.9 K: {thermodynamic_relations.compute_e_ve(T_V=jnp.array([5999.9]), species_table=species_table)[0][0]:.0f} J/kg"
)
print(
    f"T=6000.1 K: {thermodynamic_relations.compute_e_ve(T_V=jnp.array([6000.1]), species_table=species_table)[0][0]:.0f} J/kg"
)


# Convert to MJ/kg for better readability
h_species_MJ = h_species / 1e6

# Plot with Plotly
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=T,
        y=h_species_MJ,
        mode="lines",
        name=f"{species_name} (Enthalpy)",
        line=dict(color="blue", width=2),
    )
)

# Mark the temperature ranges
for T_range in T_limit_low[0, 1:]:  # Skip first one
    fig.add_vline(x=T_range, line_dash="dash", line_color="red", opacity=0.3)


fig.update_layout(
    title=f"Enthalpy vs Temperature for {species_name}",
    xaxis_title="Temperature [K]",
    yaxis_title="Vibroelectronic Enthalpy [MJ/kg]",
    hovermode="x unified",
    template="plotly_white",
    width=800,
    height=600,
)

fig.update_layout(
    title=None,
    width=800,
    height=800,
    margin=dict(t=10, b=100, l=80, r=10),
    xaxis=dict(
        title_font=dict(size=26),
        tickfont=dict(size=22),
        # tickangle=45,
    ),
    yaxis=dict(title_font=dict(size=26), tickfont=dict(size=22)),
    legend=dict(
        font=dict(size=22),
        x=0.5,
        y=-0.25,
        xanchor="center",
        yanchor="top",
        borderwidth=1,
        bordercolor="white",
        orientation="h",
    ),
)
fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/curve_fits_visualized/gnoffoenthalpyjump.pdf"
)
fig.show()

fig.update_xaxes(range=[5900, 6100])
fig.update_yaxes(range=[1.35, 1.5])
fig.show()

fig.write_image(
    "/home/hhoechter/tum/jaxfluids_internship/experiments/curve_fits_visualized/gnoffoenthalpyjump_zoomed.pdf"
)

Vibroelectronic enthalpy from Gnoffo's curve fit evaluated with repo code
T=5999.9 K: 1419069 J/kg
T=6000.1 K: 1429105 J/kg


Enthalpy at 6000 K: 1429075.0661969848 J/kg
